# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yashcodes07/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Yashcodes07/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)
df.columns.tolist()
print(df.columns.tolist())
df.head(3)

Working dir: /content/flyrank-ml-internship
(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


In [2]:
# CODE — full pipeline, inlined so this notebook is self-contained and doesn't
# depend on another notebook's runtime state or an import path.
import duckdb
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

HF_TOKEN = "hf_****************"  # TODO: your real token
REPO_ID = "FlyRank/internship-warehouse"
BASE = f"hf://datasets/{REPO_ID}"
DIM_CLIENTS_PATH = f"{BASE}/dim_clients.parquet"
DIM_CONTENT_PATH = f"{BASE}/dim_content.parquet"
DAILY_PATH = f"{BASE}/fact_content_daily_performance/*/*.parquet"
TRAIN_WINDOW_DAYS, LABEL_WINDOW_DAYS, MIN_IMPRESSIONS = 28, 28, 50

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

client_id = con.sql(f"""
    SELECT client_hash_id FROM read_parquet('{DIM_CLIENTS_PATH}')
    WHERE is_active = true AND has_gsc_access = true AND gsc_data_start IS NOT NULL
    ORDER BY gsc_data_start ASC LIMIT 1
""").df().iloc[0]["client_hash_id"]

date_range = con.sql(f"""
    SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('{DAILY_PATH}')
    WHERE client_hash_id = '{client_id}' AND gsc_data_available = true
""").df()
data_start, data_end = date_range.iloc[0]["min_date"], date_range.iloc[0]["max_date"]

def build_features(window_start, window_end):
    feats = con.sql(f"""
        WITH windowed AS (
            SELECT content_hash_id, report_date, gsc_clicks, gsc_impressions, gsc_avg_position,
                   DATE_DIFF('day', DATE '{window_start}', report_date) AS day_idx
            FROM read_parquet('{DAILY_PATH}')
            WHERE client_hash_id = '{client_id}' AND gsc_data_available = true
              AND report_date > DATE '{window_start}' AND report_date <= DATE '{window_end}'
              AND gsc_impressions > 0
        ), agg AS (
            SELECT content_hash_id, COUNT(*) AS days_observed, SUM(gsc_impressions) AS impressions_total,
                   AVG(gsc_clicks) AS clicks_mean, AVG(gsc_avg_position) AS avg_position_mean,
                   STDDEV_POP(gsc_avg_position) AS avg_position_volatility,
                   REGR_SLOPE(gsc_avg_position, day_idx) AS avg_position_slope,
                   REGR_SLOPE(gsc_clicks, day_idx) AS clicks_slope,
                   AVG(gsc_clicks) / NULLIF(AVG(gsc_impressions), 0) AS ctr_mean,
                   REGR_SLOPE(gsc_clicks / NULLIF(gsc_impressions, 0), day_idx) AS ctr_slope
            FROM windowed GROUP BY content_hash_id
        ) SELECT * FROM agg WHERE days_observed >= 5 AND impressions_total >= {MIN_IMPRESSIONS}
    """).df()
    content_dim = con.sql(f"""
        SELECT content_hash_id, content_created_date, search_volume, word_count, competition, backlinks
        FROM read_parquet('{DIM_CONTENT_PATH}')
        WHERE client_hash_id = '{client_id}' AND is_published = true
    """).df()
    feats = feats.merge(content_dim, on="content_hash_id", how="left")
    feats["page_age_days"] = (pd.Timestamp(window_end) - pd.to_datetime(feats["content_created_date"])).dt.days
    return feats.drop(columns=["content_created_date"])

def build_labels(window_start, window_end):
    df = con.sql(f"""
        WITH windowed AS (
            SELECT content_hash_id, gsc_clicks,
                   DATE_DIFF('day', DATE '{window_start}', report_date) AS day_idx
            FROM read_parquet('{DAILY_PATH}')
            WHERE client_hash_id = '{client_id}' AND gsc_data_available = true
              AND report_date > DATE '{window_start}' AND report_date <= DATE '{window_end}'
        ), agg AS (
            SELECT content_hash_id, COUNT(*) AS days_observed, AVG(gsc_clicks) AS clicks_mean,
                   REGR_SLOPE(gsc_clicks, day_idx) AS clicks_slope
            FROM windowed GROUP BY content_hash_id
        ) SELECT content_hash_id, clicks_slope / NULLIF(clicks_mean, 0) AS rel_slope
          FROM agg WHERE days_observed >= 5
    """).df()
    df["label"] = df["rel_slope"].apply(
        lambda s: "review" if pd.isna(s) else ("growing" if s > 0.02 else ("declining" if s < -0.02 else "review"))
    )
    return df[["content_hash_id", "label"]]

train_feat_start = pd.Timestamp(data_start)
train_feat_end = train_feat_start + pd.Timedelta(days=TRAIN_WINDOW_DAYS)
train_label_end = train_feat_end + pd.Timedelta(days=LABEL_WINDOW_DAYS)
test_feat_start = train_label_end
test_feat_end = test_feat_start + pd.Timedelta(days=TRAIN_WINDOW_DAYS)
test_label_end = test_feat_end + pd.Timedelta(days=LABEL_WINDOW_DAYS)

train = build_features(train_feat_start.date(), train_feat_end.date()).merge(
    build_labels(train_feat_end.date(), train_label_end.date()), on="content_hash_id")
test = build_features(test_feat_start.date(), test_feat_end.date()).merge(
    build_labels(test_feat_end.date(), test_label_end.date()), on="content_hash_id")

feature_cols = [c for c in train.columns if c not in ("content_hash_id", "label")]
X_train, y_train = train[feature_cols].fillna(0), train["label"]
X_test, y_test = test[feature_cols].fillna(0), test["label"]

baseline_score = f1_score(y_test, [y_train.mode()[0]] * len(y_test), average="macro", zero_division=0)

clf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, class_weight="balanced")
clf.fit(X_train, y_train)
preds = clf.predict(X_test)
f1_macro = f1_score(y_test, preds, average="macro", zero_division=0)

# per-row reason codes
probs = clf.predict_proba(X_test)
importances = np.array(clf.feature_importances_)
z = ((X_test - X_train.mean()) / X_train.std().replace(0, 1)).abs().values * importances
reason_codes = [", ".join([feature_cols[i] for i in np.argsort(row)[::-1][:2]]) for row in z]

ACTION_MAP = {"growing": "protect", "declining": "rewrite / improve", "review": "manual review"}
action_report = test[["content_hash_id"]].copy()
action_report["predicted_state"] = preds
action_report["confidence"] = probs.max(axis=1)
action_report["reason_code"] = reason_codes
action_report["recommended_action"] = action_report["predicted_state"].map(ACTION_MAP)
action_report = action_report.sort_values("confidence", ascending=False)

print(f"Baseline: {baseline_score:.3f}  |  Model F1-macro: {f1_macro:.3f}")
print("\nTop 15 ranked actions:")
print(action_report.head(15).to_string())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Baseline: 0.208  |  Model F1-macro: 0.380

Top 15 ranked actions:
               content_hash_id predicted_state  confidence                                  reason_code recommended_action
2483  content_0f803a460639ca9d         growing    0.743481   avg_position_mean, avg_position_volatility            protect
2590  content_f2621ec8afa053df         growing    0.732133   avg_position_volatility, avg_position_mean            protect
2531  content_ebda20cb4c29fe92          review    0.724807   avg_position_volatility, avg_position_mean      manual review
1242  content_7e7ae1bb7e05c470         growing    0.716081   avg_position_mean, avg_position_volatility            protect
1325  content_16950b05c36541cb          review    0.714679   avg_position_volatility, avg_position_mean      manual review
1331  content_06a457d16659a858          review    0.713757   avg_position_mean, avg_position_volatility      manual review
1306  content_8ba39bc32db4d6c7         growing    0.682816   avg_position

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who uses this**: a content reviewer or content lead triaging a review queue for one client's content inventory, weekly or biweekly.
What it's for: ranking which content items deserve human attention first, out of many, given limited review time — not deciding what to write or auto-publishing anything.
Where it stops being valid:

1.Trained and validated on a single client's data (see capstone Limitations). Do not apply these exact thresholds to a different client's content without re-validating — different traffic scale and content strategy will shift what "declining" looks like numerically.

2.The model's declining recall is measured at 0.15 — it misses most true declines. This playbook is not a safety net for catching every decline; it's a prioritization aid for the declines it does catch, or ones a human already suspects.

3.Trained on a specific historical window (see data_start/data_end above). If the client's content strategy, publishing cadence, or SEO landscape shifts materially, these thresholds go stale — see Section 4.
No causal claim: a declining label doesn't mean the model knows why — reason codes point to statistical drivers (e.g. avg_position_volatility), not root causes a human hasn't diagnosed yet.



In [4]:
print("Client:", client_id)
print("Training window:", data_start, "to", data_end)
print("Validated metric: F1-macro", round(f1_macro, 3), "vs baseline", round(baseline_score, 3))
print("\nKnown weak spots (from validation):")
print("- declining recall: low — model under-catches true declines")
print("- growing precision: low — model over-flags growth")


Client: client_9958f0a7ae1df715
Training window: 2025-01-27 00:00:00 to 2026-06-30 00:00:00
Validated metric: F1-macro 0.38 vs baseline 0.208

Known weak spots (from validation):
- declining recall: low — model under-catches true declines
- growing precision: low — model over-flags growth


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [5]:
# CODE — flag low-confidence items that most need human judgment before any action
low_confidence_threshold = 0.5
uncertain = action_report[action_report["confidence"] < low_confidence_threshold]
print(f"Items below {low_confidence_threshold} confidence (highest human-review priority, "
      f"least trustworthy for autonomous triage): {len(uncertain)} of {len(action_report)}")
print(uncertain.head(10).to_string())


Items below 0.5 confidence (highest human-review priority, least trustworthy for autonomous triage): 2220 of 2895
               content_hash_id predicted_state  confidence                                 reason_code recommended_action
2410  content_69451fc06bc6eb06         growing    0.499941                       ctr_mean, clicks_mean            protect
2241  content_46f188a139988b81          review    0.499877              impressions_total, clicks_mean      manual review
2082  content_a77b455c6a677cc4          review    0.499598        impressions_total, avg_position_mean      manual review
1401  content_3c0fa6e0eb80d7ee         growing    0.499116  avg_position_mean, avg_position_volatility            protect
1605  content_bbcdd929e32d8bf9         growing    0.499115              clicks_mean, impressions_total            protect
2395  content_69da140e827364c6         growing    0.499024        impressions_total, avg_position_mean            protect
2789  content_adc116ff8257a5a0  

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Signals that would tell us these recommendations have gone stale and the model needs retraining, not just re-running:

**Label distribution drift:** if the live proportion of growing/declining/review content shifts substantially from the training distribution (was review 560 / growing 488 / declining 282), the model's class-weighting assumptions no longer match reality.

**Confidence drift:** if the average prediction confidence across the queue drops meaningfully over time, the model is seeing feature patterns it wasn't trained on.

**Data staleness:** the training window is fixed (data_start–data_end above); once live data extends more than ~2x the label window (56 days) past that window, retrain on the newer window rather than trusting extrapolation.
Feature importance shift: if a re-run on fresh data shows avg_position_mean/avg_position_volatility losing their dominant share of importance, the underlying search dynamics for this client may have changed and the whole feature set should be reconsidered, not just retrained.

**Silent failure mode check:** periodically re-run the near-zero-traffic collision check from Week 9 — if the count of feature-collision rows grows, the model needs either more features or a stricter minimum-impressions floor.

In [6]:
# CODE — baseline distribution snapshot to compare future runs against
retrain_baseline = {
    "trained_on_client": client_id,
    "training_window": f"{data_start} to {data_end}",
    "label_distribution_train": train["label"].value_counts().to_dict(),
    "mean_confidence": float(action_report["confidence"].mean()),
    "top_feature_importance": dict(sorted(
        zip(feature_cols, clf.feature_importances_), key=lambda x: -x[1]
    )[:3]),
}
import json
print(json.dumps(retrain_baseline, indent=2, default=str))


{
  "trained_on_client": "client_9958f0a7ae1df715",
  "training_window": "2025-01-27 00:00:00 to 2026-06-30 00:00:00",
  "label_distribution_train": {
    "review": 560,
    "growing": 488,
    "declining": 282
  },
  "mean_confidence": 0.44973446815474727,
  "top_feature_importance": {
    "avg_position_mean": 0.14728014590507504,
    "avg_position_volatility": 0.11766910121063713,
    "impressions_total": 0.1134482074916426
  }
}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Writing the ranked queue and the retrain-trigger baseline to work/outputs/ so next week's paper can pull these files directly rather than re-running the pipeline.

In [11]:
import os
import json
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)

# 1. The ranked action queue itself
action_report.to_csv("work/outputs/ranked_action_queue.csv", index=False)

# 2. Baseline vs model comparison chart
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Baseline", "Random Forest"], [baseline_score, f1_macro], color=["#999", "#2563eb"])
ax.set_ylabel("F1-macro")
ax.set_title("Baseline vs Model — time-aware split")
plt.tight_layout()
plt.savefig("work/outputs/baseline_vs_model.png", dpi=150)
plt.close()

# 3. Retrain-trigger baseline snapshot, for comparing future runs against
with open("work/outputs/retrain_baseline.json", "w") as f:
    json.dump(retrain_baseline, f, indent=2, default=str)

# 4. Human-readable playbook summary (feeds directly into the paper's Recommendations section)
playbook_summary = f"""
CONTENT ACTION PLAYBOOK — SUMMARY
Client: {client_id}
Training window: {data_start} to {data_end}
Model: Random Forest (F1-macro {f1_macro:.3f} vs baseline {baseline_score:.3f})

Intended use: prioritize a content review queue; NOT autonomous action.
Never automate: publishing changes, bulk deprioritization, blame/performance judgments.
Retrain triggers: label distribution drift, confidence drift, data staleness (>56 days
past training window), feature importance shift, growth in feature-collision rate.
"""
with open("work/outputs/playbook_summary.txt", "w") as f:
    f.write(playbook_summary)

print("Exported to work/outputs/:")
for filename in os.listdir("work/outputs"):
    print(" -", filename)

Exported to work/outputs/:
 - playbook_summary.txt
 - retrain_baseline.json
 - ranked_action_queue.csv
 - baseline_vs_model.png


In [12]:
test_metrics = {
    "baseline_f1_macro": float(baseline_score),
    "model_f1_macro": float(f1_macro),
    "model_accuracy": float((preds == y_test).mean()),
    "relative_improvement": float((f1_macro - baseline_score) / baseline_score),
    "per_class": {
        cls: {
            "precision": float(p), "recall": float(r), "f1": float(f), "support": int(s)
        }
        for cls, p, r, f, s in zip(
            *[list(x) for x in __import__("sklearn.metrics", fromlist=["precision_recall_fscore_support"]).precision_recall_fscore_support(y_test, preds, labels=sorted(y_test.unique()))],
        )
    } if False else None,  # placeholder — see note below
}

# Cleaner version — compute per-class breakdown directly:
from sklearn.metrics import precision_recall_fscore_support
labels = sorted(y_test.unique())
precisions, recalls, f1s, supports = precision_recall_fscore_support(y_test, preds, labels=labels, zero_division=0)
test_metrics["per_class"] = {
    lbl: {"precision": float(p), "recall": float(r), "f1": float(f), "support": int(s)}
    for lbl, p, r, f, s in zip(labels, precisions, recalls, f1s, supports)
}

os.makedirs("work/results", exist_ok=True)
with open("work/results/test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)
print("Saved work/results/test_metrics.json")

Saved work/results/test_metrics.json


In [13]:
from sklearn.metrics import precision_recall_fscore_support
import json, os

labels = sorted(y_test.unique())
precisions, recalls, f1s, supports = precision_recall_fscore_support(y_test, preds, labels=labels, zero_division=0)

test_metrics = {
    "baseline_f1_macro": float(baseline_score),
    "model_f1_macro": float(f1_macro),
    "model_accuracy": float((preds == y_test.values).mean()),
    "relative_improvement": float((f1_macro - baseline_score) / baseline_score),
    "per_class": {
        lbl: {"precision": float(p), "recall": float(r), "f1": float(f), "support": int(s)}
        for lbl, p, r, f, s in zip(labels, precisions, recalls, f1s, supports)
    },
}

os.makedirs("work/results", exist_ok=True)
with open("work/results/test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)
print("Saved work/results/test_metrics.json")

Saved work/results/test_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.